<a href="https://colab.research.google.com/github/dsatish1252/sqlite-python-implementation/blob/main/Langchain_HandsOn_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U langchain langchain-google-genai google-generativeai pydantic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 15.1 MB/s eta 0:00:00


In [3]:
import os

from google.colab import userdata

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import ChatPromptTemplate

from langchain.tools import tool

from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    ToolMessage
)

from pydantic import BaseModel, Field

from typing import Literal

In [46]:
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [48]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.3
)

In [11]:
response = llm.invoke(
    "Explain customer support automation in three sentences."
)

print(response.content)

Customer support automation leverages technology to handle customer inquiries and tasks with minimal or no human intervention. This typically involves AI-powered chatbots, virtual assistants, and self-service portals that can answer common questions, process requests, or guide users through solutions. Its primary goal is to provide faster, more consistent support 24/7, while freeing human agents to focus on complex issues and improve overall operational efficiency.


In [12]:
support_prompt = ChatPromptTemplate.from_template(
"""
You are an AI customer support assistant.

Customer Name:
{customer_name}

Customer Type:
{customer_type}

Customer Query:
{customer_query}

Instructions

- Understand the customer's issue.
- Respond professionally.
- Be empathetic.
- Keep the answer concise.
- Never invent an order status.
- Use a tool whenever real-time information is required.
"""
)

In [13]:
support_chain = support_prompt | llm

In [14]:
response = support_chain.invoke(
{
    "customer_name":"Rahul",
    "customer_type":"Premium",
    "customer_query":"My order has not arrived yet."
}
)

print(response.content)

Hello Rahul,

I'm sorry to hear your order hasn't arrived yet. I understand how frustrating that can be.

To get you the most accurate and up-to-date information on your shipment, I'll need to access our order tracking system. Please bear with me for a moment while I use my tools to look into this for you.


In [15]:
class SupportTicket(BaseModel):

    category: Literal[
        "Billing",
        "Technical",
        "Account",
        "Delivery",
        "Order",
        "Refund",
        "Other"
    ]

    priority: Literal[
        "High",
        "Medium",
        "Low"
    ]

    sentiment: Literal[
        "Positive",
        "Neutral",
        "Negative"
    ]

    summary: str

    recommended_team: str

    requires_human_agent: bool

In [16]:
structured_llm = llm.with_structured_output(
    SupportTicket
)

In [18]:
ticket = structured_llm.invoke("""
I was charged twice for order ORD-1001.
Refund my money immediately.
""")

print(ticket.model_dump())

{'category': 'Refund', 'priority': 'High', 'sentiment': 'Negative', 'summary': 'Customer was double charged for order ORD-1001 and requests an immediate refund.', 'recommended_team': 'Billing', 'requires_human_agent': True}


In [20]:
orders = {

    "ORD-1001":"Shipped",

    "ORD-1002":"Processing",

    "ORD-1003":"Delivered",

    "ORD-1004":"Cancelled",

    "ORD-1005":"Out for Delivery",

    "ORD-1006":"Pending",

    "ORD-1007":"Returned",

    "ORD-1008":"Shipped",

    "ORD-1009":"Delivered",

    "ORD-1010":"Processing",

    "ORD-1011":"Shipped",

    "ORD-1012":"Cancelled",

    "ORD-1013":"Out for Delivery",

    "ORD-1014":"Delivered",

    "ORD-1015":"Pending",

    "ORD-1016":"Shipped",

    "ORD-1017":"Returned",

    "ORD-1018":"Processing",

    "ORD-1019":"Delivered",

    "ORD-1020":"Cancelled",

    "ORD-1021":"Out for Delivery",

    "ORD-1022":"Shipped",

    "ORD-1023":"Pending",

    "ORD-1024":"Delivered",

    "ORD-1025":"Returned",

    "ORD-1026":"Processing",

    "ORD-1027":"Shipped",

    "ORD-1028":"Delivered",

    "ORD-1029":"Cancelled",

    "ORD-1030":"Out for Delivery"

}

In [21]:
@tool
def get_order_status(order_id: str) -> str:
    """
    Return the status of an order.
    """

    if order_id not in orders:
        return "Order not found"

    return orders[order_id]

In [22]:
@tool
def calculate_discount(
    price: float,
    discount_percent: float
) -> float:

    """
    Calculate discounted price.
    """

    if price < 0:
        raise ValueError("Price cannot be negative.")

    if discount_percent < 0 or discount_percent > 100:
        raise ValueError(
            "Discount must be between 0 and 100."
        )

    return round(
        price * (1 - discount_percent/100),
        2
    )

In [23]:
@tool
def calculate_delivery_charge(
    order_value: float,
    customer_type: str
) -> int:

    """
    Calculate delivery charge.
    """

    if customer_type.lower()=="premium":
        return 0

    if order_value>=2000:
        return 0

    return 100

In [24]:
@tool
def get_estimated_delivery_days(
    shipping_type:str
)->str:

    """
    Estimate delivery days.
    """

    shipping={
        "standard":"3-5 business days",
        "express":"1-2 business days",
        "same day":"Same day"
    }

    key=shipping_type.lower()

    if key not in shipping:
        raise ValueError(
            "Supported shipping options are Standard, Express and Same Day."
        )

    return shipping[key]


In [25]:
tools = [

    get_order_status,

    calculate_discount,

    calculate_delivery_charge,

    get_estimated_delivery_days

]

In [26]:
llm_with_tools = llm.bind_tools(tools)

In [27]:
tool_map={

    "get_order_status":get_order_status,

    "calculate_discount":calculate_discount,

    "calculate_delivery_charge":calculate_delivery_charge,

    "get_estimated_delivery_days":get_estimated_delivery_days

}

In [33]:
history=[]

In [34]:
user_query = "What is the status of order ORD-1009"

history.append(
    HumanMessage(content=user_query)
)

In [35]:
response = llm_with_tools.invoke(history)

history.append(response)

In [36]:
if response.tool_calls:

    for tool_call in response.tool_calls:

        tool_name = tool_call["name"]

        args = tool_call["args"]

        tool = tool_map[tool_name]

        result = tool.invoke(args)

        history.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )

    final = llm_with_tools.invoke(history)

    history.append(final)

    print(final.content)

else:

    print(response.content)

The order ORD-1009 has been delivered.


In [37]:
def customer_support_assistant(user_query):

    history.append(
        HumanMessage(content=user_query)
    )

    response = llm_with_tools.invoke(history)

    history.append(response)

    if response.tool_calls:

        for tool_call in response.tool_calls:

            tool_name = tool_call["name"]

            args = tool_call["args"]

            tool = tool_map[tool_name]

            try:

                result = tool.invoke(args)

            except Exception:

                result = (
                    "Sorry, I am temporarily unable "
                    "to process your request."
                )

            history.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )

        final = llm_with_tools.invoke(history)

        history.append(final)

        print(final.content)

    else:

        print(response.content)

In [38]:
customer_support_assistant(
    "Hello"
)

Hello! How can I help you today?


In [39]:
customer_support_assistant(
    "What is the status of ORD-1002?"
)

The order ORD-1002 is currently processing.


In [40]:
customer_support_assistant(
    "Apply 20% discount to 5000"
)

A 20% discount on 5000 is 4000.


In [41]:
customer_support_assistant(
    "How long does express shipping take?"
)

Express shipping usually takes 1-2 business days.


In [42]:
customer_support_assistant(
    "Delivery charge for Standard customer with order value 1500"
)

The delivery charge for a Standard customer with an order value of 1500 is 100.


In [49]:
customer_support_assistant(
"""
Check ORD-1005 and also tell me the price
of a ₹4000 product after a 25% discount.
"""
)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 56.990716494s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [44]:
customer_support_assistant(
"""
I was charged twice for order ORD-1001.
Refund my money immediately.
"""
)

[{'type': 'text', 'text': 'I understand your concern about being charged twice for order ORD-1001. However, I am an AI assistant and cannot directly process refunds or manage billing disputes.\n\nPlease contact customer support with your order details (ORD-1001) to resolve this issue. They will be able to investigate the double charge and assist you with a refund.', 'extras': {'signature': 'CpUGARFNMg9D2fCAR/KqHy8MP8WSHahMdUqNRON6PZHB0MO3EdJ9I3beDK2gCjJVBJXKNl1aLRKCD3GUQm/SC/6rYo6qS5c5b+hFkRUKddVOiFbXCZzLSHEiYw9AGmF4CFxnGzRS08gFN5d9MakDramei4F7eBXjxXEx6rqROYW4nJchfBMBDeMAZIf+eAVahJfTFFUNhqVw/3B8xFuQZ0QmEa+E8hhMvdfYwgCzEZGa8ZHFXtLsfpOttyUIw0m26qu8vShj+hCUvNcE5Qq2TixmWisCQzmi1WTSFhPryjnFvnhpv+uQQBKsM0iDoHuFr6HTIxPjDUu3ftKPHr3zTHeLjU/6eDUOT8eAUY7KtK85QTHlmhluknOTrClqQgGWBSPeOgdUD0zt8OQplkJreX18Y+SY6Ypys2A/t2xm6haI/+n6DDuuPi9U26aBaIfBXCCchrI+OwixVyoYZdjWKzXzYsNm0eAZ5kn9dlBQSz8llMahIYusDDUA/2obi21GteMm6Hlg92737AzGhyg/xPiE2tzg98RNLwrJPEkkrhB6U3GBVEaW1h6NkX5EGcInzFwbUA1fUiu3g9qWucCRRCTceDlRyK